<a href="https://colab.research.google.com/github/deseafiles/Implementasi-dan-Evaluasi-Model-ASR-Multibahasa-untuk-Bahasa-Indonesia-dan-Bahasa-Daerah/blob/main/final_tubes_deep_learning_kel_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers accelerate evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00


In [ ]:
!pip install datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
import re
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, concatenate_datasets, Audio
from transformers import (
    AutoProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

In [ ]:
# 1. SETTINGS & MODEL ID
model_id = "openai/whisper-base"
selected_langs = ['jav', 'sun', 'ind']

In [ ]:
# 2. DATA LOADING
def load_multi_lang_ds(lang_list):
    all_ds = []
    for lang in lang_list:
        try:
            d = load_dataset("indonesian-nlp/librivox-indonesia", lang, split="train", trust_remote_code=True)
            d = d.map(lambda x: {"language": lang})
            all_ds.append(d)
        except Exception as e:
            print(f"Error loading {lang}: {e}")
    return concatenate_datasets(all_ds).shuffle(seed=42)

ds = load_multi_lang_ds(selected_langs)
ds = ds.cast_column("audio", Audio(sampling_rate=16_000))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

librivox-indonesia.py: 0.00B [00:00, ?B/s]

languages.py:   0%|          | 0.00/282 [00:00<?, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5635 [00:00<?, ? examples/s]

In [ ]:
# 3. PROCESSOR & PREPARATION
processor = AutoProcessor.from_pretrained(model_id)

def prepare_dataset(batch):
    audio = batch["audio"]
    # Extract features
    batch["input_features"] = processor(audio["array"], sampling_rate=16_000).input_features[0]

    # Clean text (Optional: remove regex part if you want to keep punctuation)
    chars_to_ignore_regex = r'[,\?\.\!\-\;\:\"\“\%\‘\”\[\]]'
    text = batch.get("sentence", "")
    clean_text = re.sub(chars_to_ignore_regex, '', text).lower()

    # Tokenize labels
    batch["labels"] = processor(text=clean_text).input_ids
    return batch

ds_processed = ds.map(prepare_dataset, remove_columns=ds.column_names, num_proc=1)
ds_split = ds_processed.train_test_split(test_size=0.2)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/6503 [00:00<?, ? examples/s]

In [ ]:
# 4. DATA COLLATOR CLASS
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
# 5. METRICS
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# 6. MODEL & TRAINING ARGUMENTS
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Force the model to use the correct language/task during generation
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-multi-id",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=100,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    gradient_checkpointing=True,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=25,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    optim="adamw_bnb_8bit"
)

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [ ]:
# 7. TRAIN
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=ds_split["train"],
    eval_dataset=ds_split["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("Memulai Training...")
trainer.train()

Memulai Training...


Epoch,Training Loss,Validation Loss,Wer
1,0.980156,0.510916,31.333220
2,0.561111,0.434053,27.441106
3,0.191085,0.416498,22.541823
4,0.090973,0.408635,22.208945
5,0.024887,0.412983,19.725162


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=1630, training_loss=0.6922795003908544, metrics={'train_runtime': 5841.5583, 'train_samples_per_second': 4.453, 'train_steps_per_second': 0.279, 'total_flos': 1.6870085001216e+18, 'train_loss': 0.6922795003908544, 'epoch': 5.0})

In [ ]:
# 8. SAVE
trainer.save_model("./whisper-id-final")
processor.save_pretrained("./whisper-id-final")
print("Model and Processor aman tersimpan!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and Processor aman tersimpan!


In [ ]:
# Switch model to evaluation mode and move to GPU
model.eval()
model.to("cuda") # or device

print("--- Hasil Testing Model ---")

# Using the first 5 samples from the original dataset (or ds_split['test'])
for i in range(5):
    sample = ds[i]

    # Pre-process audio
    input_features = processor(
        sample["audio"]["array"],
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to("cuda")

    # Generate with no_grad to save VRAM
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            # Force the model to start with the correct language token
            language="indonesian",
            task="transcribe"
        )

    # Decode
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    print(f"\n[Sample {i}]")
    print(f"Actual   : {sample['sentence']}")
    print(f"Predicted: {transcription}")

--- Hasil Testing Model ---

[Sample 0]
Actual   : dengan warna moeka jang seolah olah amat menghinakannja  fogg berkatalah dengan amarahnja  katanja
Predicted: dengan warna moeka jang seolah olah amat menghinakannja  fogg berkatalah dengan amarahnja  katanja

[Sample 1]
Actual   : ditempat lain djalan itoe seolah olah tergantoeng diatas djoerang djoerang jang amat dalam
Predicted: ditempat lain djalan itoe seolah olah tergantoeng diatas djoerang djoerang jang amat dalam

[Sample 2]
Actual   : kahar kahar jang ditarik zeboe
Predicted: kahar kahar jang ditarik zeboe

[Sample 3]
Actual   : sekarang ditangkap oléh inspecteur polisi sebagai pentjoeri
Predicted: sekarang ditangkap oléh inspecteur polisi sebagai pentjoeri

[Sample 4]
Actual   : baiklah toean fogg  saja berani bertaroeh empat riboe pond
Predicted:  baiklah toean fogg saja berani bertaroeh empat riboe pond
